<a href="https://colab.research.google.com/github/Jimpang77/Feature-Data-Trade-Off/blob/main/Reasearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score


In [7]:
import numpy as np
import pandas as pd

!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo
adult = fetch_ucirepo(id=2)
X_raw, y_raw = adult.data.features.copy(), adult.data.targets.copy()
df = X_raw.copy(); df['income'] = y_raw.iloc[:, 0].values

In [8]:
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [9]:
df.shape

(48842, 15)

In [10]:
df['income']

,income
0,<=50K
1,<=50K
2,<=50K
3,<=50K
4,<=50K
...,...
48837,<=50K.
48838,<=50K.
48839,<=50K.
48840,<=50K.


In [11]:
df.columns = [col.lower().replace('-', '_').replace('.', '_') for col in df.columns]
df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [12]:
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()
df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [13]:
df = df.replace('?', np.nan)
print(f"Missing values per column:\n{df.isna().sum()}")

Missing values per column:
age                  0
workclass         2799
fnlwgt               0
education            0
education_num        0
marital_status       0
occupation        2809
relationship         0
race                 0
sex                  0
capital_gain         0
capital_loss         0
hours_per_week       0
native_country     857
income               0
dtype: int64


In [14]:
cat_cols = df.select_dtypes(include='object').columns
df[cat_cols] = df[cat_cols].fillna('Unknown')
df.isna().sum()

,0
age,0
workclass,0
fnlwgt,0
education,0
education_num,0
marital_status,0
occupation,0
relationship,0
race,0
sex,0


In [15]:
print(f"Rows before: {len(df)}")
df = df.drop_duplicates()
print(f"Rows after: {len(df)}")

Rows before: 48842
Rows after: 48813


In [16]:
if 'education' in df.columns:
    df = df.drop(columns=['education'])
df.head()

,age,workclass,fnlwgt,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [17]:
df['income'] = df['income'].str.rstrip('.')
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})
df['income'].value_counts()

,count
income,
0,37128
1,11685


In [18]:
if 'fnlwgt' in df.columns:
    df = df.drop(columns=['fnlwgt'])
df.head()

,age,workclass,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,0
1,50,Self-emp-not-inc,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,0
2,38,Private,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,0
3,53,Private,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,0
4,28,Private,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,0


In [19]:
X = df.drop('income', axis=1)
y = df['income']

print("Preprocessing Complete according to Hands-On ML standards.")
display(X.head())
display(y.head())

Preprocessing Complete according to Hands-On ML standards.


,age,workclass,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country
0,39,State-gov,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States
1,50,Self-emp-not-inc,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States
2,38,Private,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States
3,53,Private,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States
4,28,Private,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba


,income
0,0
1,0
2,0
3,0
4,0


In [20]:
num = X.select_dtypes(exclude='object').columns.tolist()
cat = X.select_dtypes(include='object').columns.tolist()
print(num)
print(cat)

['age', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']
['workclass', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'native_country']


In [21]:
from sklearn.model_selection import train_test_split

Xtr, Xte, ytr, yte =  train_test_split(X, y, test_size=0.2, random_state=42)


## Logistic Regression

This section covers Logistic Regression models, which are used for binary classification. Performance is evaluated across three feature configurations:

1.  **4-Feature Model:** Baseline using selected numerical and categorical variables.
2.  **All Feature Model:** Utilizes all initial features from the source dataset.
3.  **All + Engineered Features Model:** Includes original features and additional derived features.

### 4-Feature Logistic Regression Model

Evaluation of a Logistic Regression model using only 'age', 'education_num', 'occupation', and 'sex' as input features.

In [22]:
num_4 = ["age", "education_num"]
cat_4 = ["occupation", "sex"]

pre_4 = ColumnTransformer([
    ("num", StandardScaler(), num_4),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_4)
])

In [23]:
pipe_4 = Pipeline([
    ("pre", pre_4),
    ("clf", LogisticRegression(max_iter=2000))
])

In [24]:
pipe_4.fit(Xtr, ytr)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'education_num']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['occupation', 'sex'])])),
                ('clf', LogisticRegression(max_iter=2000))])

In [25]:
y_pred_4 = pipe_4.predict(Xte)
y_proba_4 = pipe_4.predict_proba(Xte)[:, 1]

In [26]:
f1_4 = f1_score(yte, y_pred_4)
auc_4 = roc_auc_score(yte, y_proba_4)

print(f"4-Feature Model - F1 Score: {f1_4:.4f}")
print(f"4-Feature Model - AUC Score: {auc_4:.4f}")

4-Feature Model - F1 Score: 0.4987
4-Feature Model - AUC Score: 0.8243


### All Feature Logistic Regression Model

Evaluation of a Logistic Regression model using the complete set of original features from the preprocessed dataset.

In [27]:
Xtr[:3]

,age,workclass,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country
40819,70,Unknown,15,Divorced,Unknown,Not-in-family,White,Male,2538,0,45,United-States
40613,30,Private,8,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,44,United-States
45445,59,Private,13,Separated,Exec-managerial,Not-in-family,White,Male,0,0,45,United-States


In [28]:
ytr[:3]

,income
40819,0
40613,0
45445,0


In [29]:
pre_raw = ColumnTransformer([
    ('num', StandardScaler(), num ),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat)
])

In [30]:

pipe_raw = Pipeline( [ ('pre', pre_raw),

            ( 'clf', LogisticRegression(max_iter=2000))

            ])

In [31]:
pipe_raw.fit(Xtr, ytr)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'education_num',
                                                   'capital_gain',
                                                   'capital_loss',
                                                   'hours_per_week']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['workclass',
                                                   'marital_status',
                                                   'occupation', 'relationship',
                                                   'race', 'sex',
                                                   'native_country'])])),
                ('clf', LogisticRegression(max_iter=2000))])

### All + Engineered Features Logistic Regression Model

Evaluation of a Logistic Regression model using all original features combined with engineered features such as 'capital_net' and 'age_x_hours'.

In [32]:
def add_features(df_in):
    d = df_in.copy()
    d['capital_net'] = d['capital_gain'] - d['capital_loss']
    d['has_capital_gain'] = (d['capital_gain'] > 0).astype(int)
    d['log_capital_gain'] = np.log1p(d['capital_gain'])
    d['age_x_hours'] = d['age'] * d['hours_per_week']
    d['edu_x_hours'] = d['education_num'] * d['hours_per_week']
    d['age_sq'] = d['age'] ** 2
    return d

In [33]:
Xtr_eng = add_features(Xtr)
Xte_eng = add_features(Xte)

num_eng = Xtr_eng.select_dtypes(exclude='object').columns.tolist()
cat_eng = Xtr_eng.select_dtypes(include='object').columns.tolist()

In [34]:
pre_eng = ColumnTransformer([
    ('num', StandardScaler(), num_eng),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_eng)
])

pipe_eng = Pipeline([
    ('pre', pre_eng),
    ('clf', LogisticRegression(max_iter=2000))
])

In [35]:
pipe_eng.fit(Xtr_eng, ytr)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'education_num',
                                                   'capital_gain',
                                                   'capital_loss',
                                                   'hours_per_week',
                                                   'capital_net',
                                                   'has_capital_gain',
                                                   'log_capital_gain',
                                                   'age_x_hours', 'edu_x_hours',
                                                   'age_sq']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['workclass',
                                                   'marital_status',
                                                   'occupation', 'relationship',
                                                   'race', 'sex',
                                                   'native_country'])])),
                ('clf', LogisticRegression(max_iter=2000))])

In [36]:
y_pred_eng = pipe_eng.predict(Xte_eng)
y_proba_eng = pipe_eng.predict_proba(Xte_eng)[:, 1]

In [37]:
f1_eng = f1_score(yte, y_pred_eng)
auc_eng = roc_auc_score(yte, y_proba_eng)

print(f"Engineered Model - F1 Score: {f1_eng:.4f}")
print(f"Engineered Model - AUC Score: {auc_eng:.4f}")

Engineered Model - F1 Score: 0.6768
Engineered Model - AUC Score: 0.9101


## Decision Tree

This section implements Decision Tree classifiers. Three variations are analyzed to determine the impact of feature selection on rule-based classification performance:

1.  **4-Feature Model:** Baseline configuration using a restricted feature set.
2.  **All Feature Model:** Configuration using all original dataset features.
3.  **All + Engineered Features Model:** Configuration using original and engineered features.

In [38]:
display(Xtr.head())
display(ytr.head())

,age,workclass,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country
40819,70,Unknown,15,Divorced,Unknown,Not-in-family,White,Male,2538,0,45,United-States
40613,30,Private,8,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,44,United-States
45445,59,Private,13,Separated,Exec-managerial,Not-in-family,White,Male,0,0,45,United-States
31318,22,Private,10,Never-married,Adm-clerical,Unmarried,Black,Female,0,0,30,United-States
3719,21,Local-gov,10,Never-married,Adm-clerical,Own-child,White,Male,0,0,40,Guatemala


,income
40819,0
40613,0
45445,0
31318,0
3719,0


In [39]:
from sklearn.tree import DecisionTreeClassifier

### 4-Feature Decision Tree Model

Implementation of a Decision Tree classifier restricted to 'age', 'education_num', 'occupation', and 'sex'.

In [40]:
pre_4_dt = ColumnTransformer([
    ('num', StandardScaler(), num_4),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_4)
])

In [41]:
pipe_4_dt = Pipeline([
    ('pre', pre_4_dt),
    ('clf', DecisionTreeClassifier(max_depth=8, random_state=42))
])

In [42]:
pipe_4_dt.fit(Xtr, ytr)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'education_num']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['occupation', 'sex'])])),
                ('clf', DecisionTreeClassifier(max_depth=8, random_state=42))])

In [43]:
y_pred_4_dt = pipe_4_dt.predict(Xte)
y_proba_4_dt = pipe_4_dt.predict_proba(Xte)[:, 1]

In [44]:
f1_4_dt = f1_score(yte, y_pred_4_dt)
auc_4_dt = roc_auc_score(yte, y_proba_4_dt)

print(f"Decision Tree 4-Features - F1 Score: {f1_4_dt:.4f}")
print(f"Decision Tree 4-Features - AUC Score: {auc_4_dt:.4f}")

Decision Tree 4-Features - F1 Score: 0.5270
Decision Tree 4-Features - AUC Score: 0.8290


In [45]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, roc_auc_score

pre_4_dt = ColumnTransformer([
    ('num', StandardScaler(), num_4),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_4)
])

pipe_4_dt = Pipeline([
    ('pre', pre_4_dt),
    ('clf', DecisionTreeClassifier(max_depth=8, random_state=42))
])

pipe_4_dt.fit(Xtr, ytr)

y_pred_4_dt = pipe_4_dt.predict(Xte)
y_proba_4_dt = pipe_4_dt.predict_proba(Xte)[:, 1]

f1_4_dt = f1_score(yte, y_pred_4_dt)
auc_4_dt = roc_auc_score(yte, y_proba_4_dt)

print(f"1. Decision Tree 4-Features:")
print(f"   - F1 Score:  {f1_4_dt:.4f}")
print(f"   - AUC Score: {auc_4_dt:.4f}")

1. Decision Tree 4-Features:
   - F1 Score:  0.5270
   - AUC Score: 0.8290


### All Feature Decision Tree Model

Implementation of a Decision Tree classifier using all original features available in the dataset.

In [46]:
num

['age', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']

In [47]:
cat

['workclass',
 'marital_status',
 'occupation',
 'relationship',
 'race',
 'sex',
 'native_country']

In [48]:
pre_raw = ColumnTransformer(
    [
        ('num', StandardScaler(), num ),
        ('cat', OneHotEncoder(handle_unknown="ignore"), cat)

    ]

)

In [49]:
pi = Pipeline(
    [
        ('pre', pre_raw ),
        ( 'clf', DecisionTreeClassifier(max_depth=8))

    ]
)

In [50]:
pi.fit(Xtr, ytr)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'education_num',
                                                   'capital_gain',
                                                   'capital_loss',
                                                   'hours_per_week']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['workclass',
                                                   'marital_status',
                                                   'occupation', 'relationship',
                                                   'race', 'sex',
                                                   'native_country'])])),
                ('clf', DecisionTreeClassifier(max_depth=8))])

In [51]:
phat = pi.predict_proba(Xte)
phat

array([[0.97817008, 0.02182992],
       [0.99693703, 0.00306297],
       [0.94015748, 0.05984252],
       ...,
       [0.        , 1.        ],
       [0.5316723 , 0.4683277 ],
       [0.99693703, 0.00306297]])

In [52]:
roc_auc_score(yte, phat[:, 1])

np.float64(0.9022284018964881)

In [53]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, roc_auc_score

pre_raw_dt = ColumnTransformer([
    ('num', StandardScaler(), num),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat)
])

pipe_all_dt = Pipeline([
    ('pre', pre_raw_dt),
    ('clf', DecisionTreeClassifier(max_depth=8, random_state=42))
])

pipe_all_dt.fit(Xtr, ytr)

y_pred_all_dt = pipe_all_dt.predict(Xte)
y_proba_all_dt = pipe_all_dt.predict_proba(Xte)[:, 1]

f1_all_dt = f1_score(yte, y_pred_all_dt)
auc_all_dt = roc_auc_score(yte, y_proba_all_dt)

print(f"2. Decision Tree All-Features:")
print(f"   - F1 Score:  {f1_all_dt:.4f}")
print(f"   - AUC Score: {auc_all_dt:.4f}")

2. Decision Tree All-Features:
   - F1 Score:  0.6639
   - AUC Score: 0.9022


### All + Engineered Features Decision Tree Model

Implementation of a Decision Tree classifier incorporating both original and engineered features.

In [54]:
pre_eng_dt = ColumnTransformer([
    ('num', StandardScaler(), num_eng),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_eng)
])

In [55]:
pipe_eng_dt = Pipeline([
    ('pre', pre_eng_dt),
    ('clf', DecisionTreeClassifier(max_depth=8, random_state=42))
])

In [56]:
pipe_eng_dt.fit(Xtr_eng, ytr)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'education_num',
                                                   'capital_gain',
                                                   'capital_loss',
                                                   'hours_per_week',
                                                   'capital_net',
                                                   'has_capital_gain',
                                                   'log_capital_gain',
                                                   'age_x_hours', 'edu_x_hours',
                                                   'age_sq']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['workclass',
                                                   'marital_status',
                                                   'occupation', 'relationship',
                                                   'race', 'sex',
                                                   'native_country'])])),
                ('clf', DecisionTreeClassifier(max_depth=8, random_state=42))])

In [57]:
y_pred_eng_dt = pipe_eng_dt.predict(Xte_eng)
y_proba_eng_dt = pipe_eng_dt.predict_proba(Xte_eng)[:, 1]

In [58]:
f1_eng_dt = f1_score(yte, y_pred_eng_dt)
auc_eng_dt = roc_auc_score(yte, y_proba_eng_dt)

print(f"Decision Tree Engineered - F1 Score: {f1_eng_dt:.4f}")
print(f"Decision Tree Engineered - AUC Score: {auc_eng_dt:.4f}")

Decision Tree Engineered - F1 Score: 0.6341
Decision Tree Engineered - AUC Score: 0.9001


In [59]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, roc_auc_score

pre_eng_dt = ColumnTransformer([
    ('num', StandardScaler(), num_eng),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_eng)
])

pipe_eng_dt = Pipeline([
    ('pre', pre_eng_dt),
    ('clf', DecisionTreeClassifier(max_depth=8, random_state=42))
])

pipe_eng_dt.fit(Xtr_eng, ytr)

y_pred_eng_dt = pipe_eng_dt.predict(Xte_eng)
y_proba_eng_dt = pipe_eng_dt.predict_proba(Xte_eng)[:, 1]

f1_eng_dt = f1_score(yte, y_pred_eng_dt)
auc_eng_dt = roc_auc_score(yte, y_proba_eng_dt)

print(f"3. Decision Tree Engineered Features:")
print(f"   - F1 Score:  {f1_eng_dt:.4f}")
print(f"   - AUC Score: {auc_eng_dt:.4f}")

3. Decision Tree Engineered Features:
   - F1 Score:  0.6341
   - AUC Score: 0.9001


In [60]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

pre_4_rf = ColumnTransformer([
    ('num', StandardScaler(), num_4),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_4)
])

pipe_4_rf = Pipeline([
    ('pre', pre_4_rf),
    ('clf', RandomForestClassifier(random_state=42))
])

pipe_4_rf.fit(Xtr, ytr)

y_pred_4_rf = pipe_4_rf.predict(Xte)
y_proba_4_rf = pipe_4_rf.predict_proba(Xte)[:, 1]

f1_4_rf = f1_score(yte, y_pred_4_rf)
auc_4_rf = roc_auc_score(yte, y_proba_4_rf)

print(f"1. Random Forest 4-Features:")
print(f"   - F1 Score:  {f1_4_rf:.4f}")
print(f"   - AUC Score: {auc_4_rf:.4f}")

1. Random Forest 4-Features:
   - F1 Score:  0.5116
   - AUC Score: 0.7977


### Model Performance Summary

The following metrics summarize the performance of all models tested. Metrics include F1 Score and Area Under the ROC Curve (AUC).

#### Logistic Regression
*   **4-Feature:** F1: 0.4987, AUC: 0.8243
*   **All + Engineered:** F1: 0.6768, AUC: 0.9101

#### Decision Tree
*   **4-Feature:** F1: 0.5270, AUC: 0.8290
*   **All Features:** F1: 0.6639, AUC: 0.9022
*   **All + Engineered:** F1: 0.6341, AUC: 0.9001

#### Random Forest
*   **4-Feature:** F1: 0.5116, AUC: 0.7977
*   **All Features:** F1: 0.6598, AUC: 0.8881
*   **All + Engineered:** F1: 0.6662, AUC: 0.8946

### Error Analysis: df.shape()

The `TypeError: 'tuple' object is not callable` occurred because `df.shape` is a tuple attribute containing the dimensions of the DataFrame. In Python, adding `()` after an attribute attempts to execute it as a function. Since a tuple cannot be called, the interpreter throws an error. Correct usage is `df.shape` without parentheses.

## Random Forest

This section evaluates Random Forest ensemble models. The following three configurations are tested:

1.  **4-Feature Model:** Ensemble baseline with minimal features.
2.  **All Feature Model:** Ensemble using all original features.
3.  **All + Engineered Features Model:** Ensemble using all available original and engineered features.

### 4-Feature Random Forest Model

Application of a Random Forest classifier using 'age', 'education_num', 'occupation', and 'sex'.

In [61]:
num_4 = ['age', 'education_num']
cat_4 = ['occupation', 'sex']

In [62]:
pre_4_rf = ColumnTransformer([
    ('num', StandardScaler(), num_4),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_4)
])

In [63]:
pipe_4_rf = Pipeline([
    ('pre', pre_4_rf),
    ('clf', RandomForestClassifier())
])

In [64]:
pipe_4_rf.fit(Xtr, ytr)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'education_num']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['occupation', 'sex'])])),
                ('clf', RandomForestClassifier())])

In [65]:
y_pred_4_rf = pipe_4_rf.predict(Xte)
y_proba_4_rf = pipe_4_rf.predict_proba(Xte)[:, 1]

In [66]:
f1_4_rf = f1_score(yte, y_pred_4_rf)
auc_4_rf = roc_auc_score(yte, y_proba_4_rf)

print(f"Random Forest 4-Features - F1 Score: {f1_4_rf:.4f}")
print(f"Random Forest 4-Features - AUC Score: {auc_4_rf:.4f}")

Random Forest 4-Features - F1 Score: 0.5121
Random Forest 4-Features - AUC Score: 0.7982


In [67]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

pre_raw_rf = ColumnTransformer([
    ('num', StandardScaler(), num),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat)
])

pipe_all_rf = Pipeline([
    ('pre', pre_raw_rf),
    ('clf', RandomForestClassifier(random_state=42))
])

pipe_all_rf.fit(Xtr, ytr)

y_pred_all_rf = pipe_all_rf.predict(Xte)
y_proba_all_rf = pipe_all_rf.predict_proba(Xte)[:, 1]

f1_all_rf = f1_score(yte, y_pred_all_rf)
auc_all_rf = roc_auc_score(yte, y_proba_all_rf)

print(f"2. Random Forest All-Features:")
print(f"   - F1 Score:  {f1_all_rf:.4f}")
print(f"   - AUC Score: {auc_all_rf:.4f}")

2. Random Forest All-Features:
   - F1 Score:  0.6598
   - AUC Score: 0.8881


### All Feature Random Forest Model

Application of a Random Forest classifier using all original features from the dataset.

In [68]:
from sklearn.ensemble import RandomForestClassifier

In [69]:
pre_raw = ColumnTransformer(
    [
        ( 'num', StandardScaler(), num),
        ( 'cat', OneHotEncoder(handle_unknown="ignore"), cat),

    ]
)

In [70]:
pipe = Pipeline(
    [
        ('pre', pre_raw ),
        ( 'clf', RandomForestClassifier() ),
    ]
)

In [71]:
pipe.fit(Xtr, ytr)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'education_num',
                                                   'capital_gain',
                                                   'capital_loss',
                                                   'hours_per_week']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['workclass',
                                                   'marital_status',
                                                   'occupation', 'relationship',
                                                   'race', 'sex',
                                                   'native_country'])])),
                ('clf', RandomForestClassifier())])

In [72]:
phat = pipe.predict_proba(Xte)[:, 1]
phat

array([0.06      , 0.        , 0.02      , ..., 1.        , 0.10569048,
       0.        ])

In [73]:
roc_auc_score(yte, phat)

np.float64(0.8877514124293785)

### All + Engineered Features Random Forest Model

Application of a Random Forest classifier using all original and engineered features.

In [74]:
pre_eng_rf = ColumnTransformer([
    ("num", StandardScaler(), num_eng),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_eng)
])

In [75]:
pipe_eng_rf = Pipeline([
    ("pre", pre_eng_rf),
    ("clf", RandomForestClassifier())
])

In [76]:
pipe_eng_rf.fit(Xtr_eng, ytr)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'education_num',
                                                   'capital_gain',
                                                   'capital_loss',
                                                   'hours_per_week',
                                                   'capital_net',
                                                   'has_capital_gain',
                                                   'log_capital_gain',
                                                   'age_x_hours', 'edu_x_hours',
                                                   'age_sq']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['workclass',
                                                   'marital_status',
                                                   'occupation', 'relationship',
                                                   'race', 'sex',
                                                   'native_country'])])),
                ('clf', RandomForestClassifier())])

In [77]:
y_pred_eng_rf = pipe_eng_rf.predict(Xte_eng)
y_proba_eng_rf = pipe_eng_rf.predict_proba(Xte_eng)[:, 1]

In [78]:
f1_eng_rf = f1_score(yte, y_pred_eng_rf)
auc_eng_rf = roc_auc_score(yte, y_proba_eng_rf)

print(f"Random Forest Engineered - F1 Score: {f1_eng_rf:.4f}")
print(f"Random Forest Engineered - AUC Score: {auc_eng_rf:.4f}")

Random Forest Engineered - F1 Score: 0.6719
Random Forest Engineered - AUC Score: 0.8959


In [79]:
pre_eng_rf = ColumnTransformer([
    ("num", StandardScaler(), num_eng),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_eng)
])

pipe_eng_rf = Pipeline([
    ("pre", pre_eng_rf),
    ("clf", RandomForestClassifier(random_state=42))
])

pipe_eng_rf.fit(Xtr_eng, ytr)

y_pred_eng_rf = pipe_eng_rf.predict(Xte_eng)
y_proba_eng_rf = pipe_eng_rf.predict_proba(Xte_eng)[:, 1]

f1_eng_rf = f1_score(yte, y_pred_eng_rf)
auc_eng_rf = roc_auc_score(yte, y_proba_eng_rf)

print(f"3. Random Forest Engineered Features:")
print(f"   - F1 Score:  {f1_eng_rf:.4f}")
print(f"   - AUC Score: {auc_eng_rf:.4f}")

3. Random Forest Engineered Features:
   - F1 Score:  0.6662
   - AUC Score: 0.8946


### Final Results Overview

This summary provides a comparison of the F1 scores and AUC (Area Under Curve) for the three classification algorithms used: Logistic Regression, Decision Trees, and Random Forests.

*   **4-Feature Baseline:** This version establishes the minimum expected performance using only basic demographic features.
*   **All Features:** This version utilizes the full original dataset to capture more complexity.
*   **Engineered Features:** This version tests whether creating new features (like net capital or interaction terms) provides additional predictive power.

In general, incorporating all features and specifically adding engineered features tended to increase the AUC scores across all model types, indicating better overall classification capability.